In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(f'Tensorflow version: {tf.version.VERSION}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))


Num GPUs Available:  0
Tensorflow version: 2.21.0
Using device:  cuda
NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub

from src.dataset.out_conv_dataset.OutConvDs import OutConvDs
from src.model.MultiU_NetModel import OutLayer
from src.training.TrainSession import train_session
from src.dataset.CheckDataset import check_dataset


当前脚本路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel\src\dataset
项目根路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel


In [3]:
def print_model_params(model):
    print("Model parameter summary:")
    print("-" * 60)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            numel = param.numel()
            total += numel
            print(f"{name:<50} {numel:>12,}")
    print("-" * 60)
    print(f"{'Total trainable parameters':<50} {total:>12,}")


In [4]:
# 检查硬件环境
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [5]:
from torchvision import transforms
import os

para_dir = 'resources/para'

para_list = [
    os.path.join(para_dir, 'class0_phase5.pth'),
    os.path.join(para_dir, 'class1_phase5.pth'),
    os.path.join(para_dir, 'class2_phase5.pth'),
    os.path.join(para_dir, 'class3_phase5.pth'),
    os.path.join(para_dir, 'class4_phase5.pth'),
]

# 加载数据集
ds_path = check_dataset()
dataset = OutConvDs(
    image_dir='drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    label_dir='drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    model_param_list=para_list,
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]),
    # data_enforcement=True,
)

from torch.utils.data import random_split, DataLoader

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=2, shuffle=True)

print(f"数据集总样本数: {len(dataset)}")
print(f"训练集样本数: {len(train_set)}")
print(f"测试集样本数: {len(test_set)}")

img0, lbl0, _ = train_set.__getitem__(0)
print(f"Train set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")


Dataset already exists.
成功匹配文件: 305.png, 321.png, 579.png 等 400 个文件
数据集总样本数: 400
训练集样本数: 360
测试集样本数: 40
Train set - img shape: torch.Size([10, 736, 960]), lbl shape: torch.Size([736, 960])
Train set - unique labels in lbl0: tensor([0, 2, 3, 4])


In [ ]:
from src.dataset.DroneSegDataSet import convert_to_binary_label

criterion = torch.nn.CrossEntropyLoss()
start_phase = 0

model = OutLayer().to(device)

train_session(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    title=f"Out Layer Training",
    device=device,
    start_phase=start_phase,
)
